In [ ]:
# Verify T4 is actually allocated before anything else runs.
# Colab free tier sometimes silently drops to CPU-only.
!nvidia-smi

# reconcile_gst2b_env — real Qwen2.5-3B baseline (T4, inference-only)

Replaces `data/baseline_metrics_real.json` (currently a placeholder) with
honest real numbers from Qwen2.5-3B-Instruct on 30 heldout seeds × 2
conditions (raw / prompted) × 3 samples = 180 rollouts. Target ≤60 min on
T4.

If T4 is oversubscribed and you want a faster run, use
`--n-seeds=20 --n-samples=2` for 80 rollouts instead.

## 1 — Install deps (inference-only; no Unsloth, no TRL, no xformers pin)

In [ ]:
# Inference-only: just transformers + accelerate (+ bitsandbytes for optional
# 4/8-bit; not used by this script but keeps Qwen loaders happy).
# We intentionally do NOT install unsloth or trl or pin xformers — those were
# required for the training dry-run and contributed to the Colab wheel hell
# we fixed in commit 92e7c85. Inference doesn't need them.
!pip install -q transformers accelerate bitsandbytes
!pip install -q networkx pandas numpy matplotlib "pydantic>=2.5"

## 2 — Clone repo (force-refresh) and install openenv-core

In [ ]:
import os, sys

# Fork URL — contains the scaffold/reconcile-gst2b branch.
REPO_URL = os.environ.get('REPO_URL', 'https://github.com/akashkathole7/OpenEnv.git')
BRANCH = os.environ.get('REPO_BRANCH', 'scaffold/reconcile-gst2b')

# Step out of /content/OpenEnv BEFORE wiping it — otherwise the shell's cwd
# gets yanked out from under subsequent !-commands.
%cd /content
!rm -rf /content/OpenEnv
!git clone --branch {BRANCH} --depth 1 {REPO_URL} /content/OpenEnv
%cd /content/OpenEnv

# Installs openenv-core + all its transitive deps (fastmcp, fastapi, ...).
!pip install -q -e /content/OpenEnv

sys.path.insert(0, '/content/OpenEnv')
sys.path.insert(0, '/content/OpenEnv/src')

# Confirm the real_baseline fix is loaded.
!cd /content/OpenEnv && git log --oneline -3
print()
!ls /content/OpenEnv/envs/reconcile_gst2b_env/scripts/real_baseline.py && echo 'real_baseline.py present'

## 3 — Run real baseline

Default: 30 seeds × 3 samples × 2 conditions = 180 rollouts.
Fallback if over budget: change `--n-seeds=30 --n-samples=3` → `--n-seeds=20 --n-samples=2`.

In [ ]:
!PYTHONPATH=/content/OpenEnv/src:/content/OpenEnv python -m envs.reconcile_gst2b_env.scripts.real_baseline \
    --condition=both --n-seeds=30 --n-samples=3 \
    --output=/content/OpenEnv/data/baseline_metrics_real.json

## 4 — Plot raw vs prompted + print delta with 95% CI

In [ ]:
import json
import matplotlib.pyplot as plt

with open('/content/OpenEnv/data/baseline_metrics_real.json') as f:
    m = json.load(f)

raw_totals = [r['total'] for r in m['raw']['rollouts']]
prompted_totals = [r['total'] for r in m['prompted']['rollouts']]
delta = m['delta_prompted_minus_raw']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
bins = 20
axes[0].hist(raw_totals, bins=bins, alpha=0.6, label=f"raw (mean {m['raw']['summary']['total_mean']:.3f})", color='#d73a49')
axes[0].hist(prompted_totals, bins=bins, alpha=0.6, label=f"prompted (mean {m['prompted']['summary']['total_mean']:.3f})", color='#2188ff')
axes[0].set_title('total reward distribution — raw vs prompted')
axes[0].set_xlabel('composite total')
axes[0].legend()
axes[0].grid(alpha=0.3)

components = ('R1', 'R2', 'R3', 'R4')
x = range(len(components))
raw_means = [m['raw']['summary']['component_summary'][c]['mean'] for c in components]
prompted_means = [m['prompted']['summary']['component_summary'][c]['mean'] for c in components]
width = 0.35
axes[1].bar([i - width/2 for i in x], raw_means, width, label='raw', color='#d73a49', alpha=0.8)
axes[1].bar([i + width/2 for i in x], prompted_means, width, label='prompted', color='#2188ff', alpha=0.8)
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(components)
axes[1].set_title('per-component means')
axes[1].set_ylabel('mean component reward')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/content/OpenEnv/data/baseline_metrics_real.png', dpi=120)
plt.show()

print(f"\n=== delta prompted − raw = {delta['mean']:.4f}  (CI95 {delta['ci95']}) ===")
print(f"passes gate (≥ 0.05): {delta['passes_gate_05']}")
print(f"\nraw catastrophic_corruption_rate: {m['raw']['summary']['catastrophic_corruption_rate']}")
print(f"prompted catastrophic_corruption_rate: {m['prompted']['summary']['catastrophic_corruption_rate']}")

## 5 — Commit instructions

Download `baseline_metrics_real.json` to your laptop, drop it into `data/`, commit:

```bash
cd /path/to/ReconcileEnv-GST2B
# scp or drag-and-drop from Colab file browser to local data/
git add data/baseline_metrics_real.json
git commit -m "data: real Qwen2.5-3B baseline — delta prompted−raw = X.XXX"
git push fork scaffold/reconcile-gst2b
```

Then paste the real numbers back to Claude; it will update README.md and BLOG.md to cite them instead of the mock baseline.

In [ ]:
# Convenience: download the JSON from Colab.
from google.colab import files  # type: ignore
files.download('/content/OpenEnv/data/baseline_metrics_real.json')